# Limpeza da base (antes de qualquer análise)

Regras desta etapa:
1. Remover duplicatas **exatas**
2. Corrigir tipos / unidades / strings
3. Descartar linhas totalmente ruins (se houver)
4. Marcar valores impossíveis como faltando (`NaN`)
5. **Não** imputar e **não** remover outliers aqui

In [ ]:
import numpy as np
import pandas as pd

# Coleta dos dados: funciona no Colab e localmente
# 1) usa o arquivo se já existir (upload no Colab em /content, ou files/ no projeto)
# 2) senão baixa do GitHub (repositório público)
from pathlib import Path
import urllib.request

URL_DATASET = 'https://raw.githubusercontent.com/abdielpaulino/dxf-nesting-time/main/files/dataset.xlsx'
CANDIDATOS = [Path('/content/dataset.xlsx'), Path('files/dataset.xlsx'), Path('dataset.xlsx')]

caminho = next((p for p in CANDIDATOS if p.exists()), None)
if caminho is None:
    caminho = Path('dataset.xlsx')
    print('Baixando dataset do GitHub...')
    urllib.request.urlretrieve(URL_DATASET, caminho)

df_raw = pd.read_excel(caminho, sheet_name='Matriz_Completa')   # cópia intacta dos dados brutos
df = df_raw.copy()
log = {'linhas_iniciais': len(df)}
print(f'Lido de: {caminho} -> {df.shape[0]:,} linhas x {df.shape[1]} colunas')
df.head(3)

## 1. Duplicatas exatas

In [ ]:
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
log['duplicatas_exatas_removidas'] = int(n_dup)
print(f'{n_dup} duplicatas exatas removidas -> {len(df)} linhas')

## 2. Tipos, unidades e strings

- Strings: remove espaços nas pontas e espaços duplicados (sem mudar caixa/acentos, que são categorias do domínio).
- `Espessura` vem como texto (`'0,65 mm'`, `'3,18 mm (1/8")'`): mantém o texto original e cria `Espessura (mm)` numérica (vírgula decimal -> ponto).
- Colunas numéricas forçadas para `float` (texto não convertível vira `NaN`).
- Tempos em segundos e em minutos: o dataset traz as duas unidades, então conferimos a consistência.

In [ ]:
COLS_TEXTO = ['Material', 'Espessura', 'Gás Auxiliar', 'Tipo Bocal', 'Tipo de Acabamento', 'Plano Desenho']
COLS_NUM = [
    'Potência Laser (W)', 'Pressão Gás (bar)', 'Velocidade Corte (m/min)', 'Posição Foco (mm)',
    'Diâmetro Bocal (mm)', 'Distância Bocal (mm)', 'Perimetro (mm)',
    'Tempo Corte Estimado Segundos', 'Tempo Real Segundos',
    'Tempo Corte Estimado Minutos', 'Tempo Real Minutos', 'Diferença de Tempo Minutos',
]

for c in COLS_TEXTO:
    df[c] = (
        df[c].astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .replace('', pd.NA)
    )

# Texto numérico com vírgula decimal -> ponto, depois float
for c in COLS_NUM:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype('string').str.strip().str.replace(',', '.', regex=False)
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')

# Espessura numérica em mm: '3,18 mm (1/8")' -> 3.18
df['Espessura (mm)'] = pd.to_numeric(
    df['Espessura'].str.extract(r'^\s*([\d]+(?:[.,]\d+)?)\s*mm', expand=False).str.replace(',', '.'),
    errors='coerce',
).astype('float64')

df[COLS_TEXTO] = df[COLS_TEXTO].astype('category')
df.dtypes

In [ ]:
# Consistência de unidades (segundos x minutos) - só diagnóstico, não altera nada
print('max |seg/60 - min| real     :', (df['Tempo Real Segundos'] / 60 - df['Tempo Real Minutos']).abs().max())
print('max |seg/60 - min| estimado :', (df['Tempo Corte Estimado Segundos'] / 60 - df['Tempo Corte Estimado Minutos']).abs().max())
print('Espessura sem número        :', df['Espessura (mm)'].isna().sum())

## 3. Valores impossíveis -> `NaN`

Regras físicas/de domínio (limites **fisicamente impossíveis**, não estatísticos — outlier plausível é mantido):

| Coluna | Impossível se |
|---|---|
| Potência, Velocidade, Perímetro, Diâmetro do bocal, Espessura | `<= 0` |
| Pressão do gás, Distância do bocal | `< 0` |
| Tempos (real, estimado, s e min) | `<= 0` |
| Qualquer coluna numérica | `inf` |

`Posição Foco` pode ser negativa (foco abaixo da superfície), então não entra na regra.

In [ ]:
df[COLS_NUM + ['Espessura (mm)']] = df[COLS_NUM + ['Espessura (mm)']].replace([np.inf, -np.inf], np.nan)

POSITIVO_ESTRITO = [
    'Potência Laser (W)', 'Velocidade Corte (m/min)', 'Perimetro (mm)', 'Diâmetro Bocal (mm)', 'Espessura (mm)',
    'Tempo Corte Estimado Segundos', 'Tempo Real Segundos',
    'Tempo Corte Estimado Minutos', 'Tempo Real Minutos',
]
NAO_NEGATIVO = ['Pressão Gás (bar)', 'Distância Bocal (mm)']  # 'Diferença de Tempo Minutos' (real - estimado) pode ser negativa

impossiveis = {}
for c in POSITIVO_ESTRITO:
    mask = df[c] <= 0
    impossiveis[c] = int(mask.sum())
    df.loc[mask, c] = np.nan
for c in NAO_NEGATIVO:
    mask = df[c] < 0
    impossiveis[c] = int(mask.sum())
    df.loc[mask, c] = np.nan

log['valores_impossiveis_marcados_nan'] = impossiveis
pd.Series(impossiveis, name='impossíveis -> NaN')

## 4. Linhas totalmente ruins

Descartada a linha se:
- o alvo (`Tempo Real Minutos`) está faltando — não dá para treinar nem avaliar; ou
- a linha não identifica o desenho (`Plano Desenho`) nem tem `Perimetro (mm)`; ou
- **todas** as colunas numéricas estão faltando.

Linhas com apenas alguns `NaN` são **mantidas** (a decisão de imputar fica para a etapa seguinte).

In [ ]:
sem_alvo = df['Tempo Real Minutos'].isna()
sem_geometria = df['Plano Desenho'].isna() & df['Perimetro (mm)'].isna()
tudo_nan = df[COLS_NUM].isna().all(axis=1)

ruins = sem_alvo | sem_geometria | tudo_nan
log['linhas_totalmente_ruins_descartadas'] = int(ruins.sum())
df = df.loc[~ruins].reset_index(drop=True)
print(f'{ruins.sum()} linhas descartadas -> {len(df)} linhas')

## Resumo da limpeza

In [ ]:
log['linhas_finais'] = len(df)
print(pd.Series(log, dtype=object).to_string())
print()
print('NaN por coluna (apenas colunas com falta):')
faltas = df.isna().sum()
print(faltas[faltas > 0].to_string() if (faltas > 0).any() else 'nenhum NaN restante')
print()
df.info()

In [ ]:
df_limpo = df  # base limpa, sem imputação e sem remoção de outliers
df_limpo.head()

# Split treino / teste

Split **por desenho** (`Plano Desenho`): cada desenho aparece em ~406 linhas (uma por combinação de parâmetros), então o mesmo desenho não pode estar no treino e no teste. 80% dos desenhos para treino, 20% para teste. O teste não é usado para nenhuma decisão.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

grupos = df_limpo['Plano Desenho'].astype(str)
idx_treino, idx_teste = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(df_limpo, groups=grupos))

df_treino = df_limpo.iloc[idx_treino].copy()
df_teste = df_limpo.iloc[idx_teste].copy()

assert not set(grupos.iloc[idx_treino]) & set(grupos.iloc[idx_teste]), 'desenho repetido entre treino e teste'
for nome, d in [('treino', df_treino), ('teste', df_teste)]:
    print(f'{nome:6s} linhas={len(d):>7,}  desenhos={d["Plano Desenho"].nunique():>4}  ({len(d)/len(df_limpo):.0%})')

# EDA (somente no treino)

Tudo daqui em diante usa **`df_treino`**. O `df_teste` não é lido: nenhum limiar, escolha de feature ou decisão sai dele.

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

from cycler import cycler
from pathlib import Path

# paleta validada (azul/laranja/verde-água; vermelho só para polaridade negativa; cinza para a referência)
COR = {'azul': '#2a78d6', 'laranja': '#eb6834', 'verde': '#1baf7a', 'vermelho': '#e34948',
       'cinza': '#8a8984', 'texto': '#0b0b0b', 'texto2': '#52514e', 'grade': '#e6e5e1', 'neutro': '#f0efec'}
plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 150, 'savefig.bbox': 'tight', 'figure.facecolor': 'white',
    'axes.prop_cycle': cycler(color=[COR['azul'], COR['laranja'], COR['verde']]),
    'axes.grid': True, 'grid.color': COR['grade'], 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': COR['cinza'],
    'axes.labelcolor': COR['texto2'], 'xtick.color': COR['texto2'], 'ytick.color': COR['texto2'],
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
})
PASTA_FIG = Path('figuras'); PASTA_FIG.mkdir(exist_ok=True)

def salvar(fig, nome):
    fig.savefig(PASTA_FIG / f'{nome}.png')
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')

eda = df_treino.copy()
ALVO = 'Tempo Real Minutos'
PARAMS_NUM = ['Potência Laser (W)', 'Pressão Gás (bar)', 'Velocidade Corte (m/min)', 'Posição Foco (mm)',
              'Diâmetro Bocal (mm)', 'Distância Bocal (mm)', 'Espessura (mm)']
GEOMETRIA = ['Perimetro (mm)']
CATEGORICAS = ['Material', 'Gás Auxiliar', 'Tipo Bocal', 'Tipo de Acabamento']
# colunas que carregam o alvo ou a fórmula (analisadas na seção de leakage)
SUSPEITAS = ['Tempo Real Segundos', 'Tempo Corte Estimado Segundos', 'Tempo Corte Estimado Minutos', 'Diferença de Tempo Minutos']

# feature derivada candidata: tempo teórico = perímetro / velocidade
eda['tempo_teorico_min'] = eda['Perimetro (mm)'] / (eda['Velocidade Corte (m/min)'] * 1000)
eda['razao_real_teorico'] = eda[ALVO] / eda['tempo_teorico_min']
print(eda.shape, '| desenhos:', eda['Plano Desenho'].nunique())

## 1. Distribuições

In [ ]:
NUM = PARAMS_NUM + GEOMETRIA + [ALVO, 'tempo_teorico_min']
eda[NUM].describe(percentiles=[.01, .25, .5, .75, .99]).T.assign(
    assimetria=eda[NUM].skew(), nunique=eda[NUM].nunique())

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
ax = ax.ravel()
ax[0].hist(eda[ALVO], bins=100);                       ax[0].set_title('Alvo: Tempo Real (min)')
ax[1].hist(np.log10(eda[ALVO]), bins=100);             ax[1].set_title('Alvo: log10(Tempo Real)')
ax[2].hist(np.log10(eda['Perimetro (mm)']), bins=80);  ax[2].set_title('log10(Perímetro mm)')
ax[3].hist(np.log10(eda['Velocidade Corte (m/min)']), bins=80); ax[3].set_title('log10(Velocidade m/min)')
ax[4].hist(eda['Potência Laser (W)'], bins=40);        ax[4].set_title('Potência (W)')
ax[5].hist(np.log10(eda['razao_real_teorico']), bins=100); ax[5].set_title('log10(Real / Teórico)')
plt.tight_layout(); plt.show()
print(f"assimetria do alvo: {eda[ALVO].skew():.2f}  ->  em log: {np.log(eda[ALVO]).skew():.2f}")

**O que cada gráfico mostra**

1. **Alvo em minutos:** quase tudo se concentra na primeira barra, com cauda até ~300 min. No erro quadrático (RMSE), poucas peças enormes dominam e as 95% pequenas são ignoradas.
2. **`log10(Tempo Real)`:** o log resolve boa parte: pico em torno de −0,6 (≈0,25 min) e assimetria de 11,9 para 1,26. Esse deve ser o alvo.
3. **`log10(Perímetro)`:** serrilhado porque só existem 348 perímetros distintos (um por desenho, repetido 406 vezes). Há poucos desenhos grandes (20 mil a 130 mil mm), região onde o modelo tem menos exemplos.
4. **`log10(Velocidade)`:** também serrilhado (só 406 combinações de parâmetros), com formato de sino e cauda leve até ~200 m/min.
5. **Potência:** só 4 valores (1500, 3000, 6000 e 12000 W); na prática é uma variável categórica ordinal.
6. **`log10(Real / Teórico)`:** o real nunca é menor que 1,29× o teórico (perímetro ÷ velocidade). O pico está em ~1,5× e a cauda chega a ~1900×, o que sugere `real ≈ 1,5 × teórico + tempo fixo por peça`, com o tempo fixo dominando nas peças pequenas.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
for a, c in zip(ax, CATEGORICAS):
    eda[c].astype(str).value_counts().sort_values().plot.barh(ax=a); a.set_title(c); a.set_ylabel('')
plt.tight_layout(); plt.show()

# desenhos: cada um deve aparecer em ~406 linhas (uma por combinação de parâmetros)
linhas_por_desenho = eda.groupby('Plano Desenho', observed=True).size()
print('linhas por desenho:', linhas_por_desenho.describe()[['min', '50%', 'max']].to_dict())
print('perímetro único por desenho:', (eda.groupby('Plano Desenho', observed=True)['Perimetro (mm)'].nunique() == 1).all())

## 2. Padrões de missing

In [ ]:
faltas = eda.isna().sum().to_frame('n_nan').assign(pct=lambda d: d['n_nan'] / len(eda) * 100)
print('linhas com algum NaN:', int(eda.isna().any(axis=1).sum()))
faltas[faltas['n_nan'] > 0] if (faltas['n_nan'] > 0).any() else print('Nenhum NaN no treino.')

In [ ]:
# Missing "disfarçado": valores sentinela / repetidos suspeitos nas colunas numéricas
sentinelas = {}
for c in PARAMS_NUM + GEOMETRIA + [ALVO]:
    top = eda[c].value_counts(normalize=True).head(1)
    sentinelas[c] = {'valor_mais_comum': top.index[0], 'pct_linhas': top.iloc[0] * 100}
pd.DataFrame(sentinelas).T

In [ ]:
# Se houvesse NaN, este bloco mostraria se faltam juntos / dependem de alguma categoria
if eda.isna().any().any():
    m = eda.isna().astype(int)
    display(m.corr().loc[lambda d: d.abs().sum(axis=1) > 1])
    for c in CATEGORICAS:
        display(m.groupby(eda[c], observed=True).mean().loc[:, m.sum() > 0])
else:
    print('Sem NaN: não há padrão de missing (MCAR/MAR/MNAR) a investigar.')

## 3. Balanceamento, outliers e limiares de winsorização/clipping

O problema é de **regressão**, então não há "classe" para balancear. O equivalente é checar se o **alvo está bem coberto** pelas categorias e faixas (e quão assimétrico ele é).

Aqui só **medimos**; nada é cortado. Os limiares saem exclusivamente do treino e serão aplicados depois (treino e teste usam os limites do treino).

In [ ]:
# Cobertura do alvo por categoria (equivalente ao balanceamento de classes)
eda['faixa_alvo'] = pd.qcut(eda[ALVO], 5, labels=['q1 (menor)', 'q2', 'q3', 'q4', 'q5 (maior)'])
for c in ['Material', 'Gás Auxiliar']:
    display((pd.crosstab(eda[c], eda['faixa_alvo'], normalize='index') * 100).round(1))
display(eda.groupby('Material', observed=True)[ALVO].describe()[['count', 'mean', '50%', 'max']])

In [ ]:
def tabela_outliers(d, cols, log=False):
    linhas = {}
    for c in cols:
        x = np.log10(d[c]) if log else d[c]
        q1, q3 = x.quantile([.25, .75]); iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        if log: lo, hi = 10 ** lo, 10 ** hi
        linhas[c] = {'p0.5': d[c].quantile(.005), 'p1': d[c].quantile(.01), 'p50': d[c].median(),
                     'p99': d[c].quantile(.99), 'p99.5': d[c].quantile(.995), 'max': d[c].max(),
                     'cerca_inf_IQR': lo, 'cerca_sup_IQR': hi,
                     '%acima_cerca': (d[c] > hi).mean() * 100, '%abaixo_cerca': (d[c] < lo).mean() * 100}
    return pd.DataFrame(linhas).T

print('Cercas de Tukey (1.5*IQR) na escala LOG (a escala natural é muito assimétrica):')
tabela_outliers(eda, [ALVO, 'Perimetro (mm)', 'Velocidade Corte (m/min)', 'tempo_teorico_min'], log=True)

In [ ]:
# Efeito de cada limiar candidato de clipping/winsorização (sem aplicar)
linhas = []
for c in [ALVO, 'Perimetro (mm)']:
    for lo_q, hi_q in [(0, .99), (0, .995), (0, .999), (.01, .99)]:
        lo, hi = eda[c].quantile(lo_q), eda[c].quantile(hi_q)
        cl = eda[c].clip(lo, hi)
        linhas.append({'coluna': c, 'quantis': f'{lo_q:.3f}-{hi_q:.3f}', 'limite_inf': lo, 'limite_sup': hi,
                       'linhas_afetadas_%': ((eda[c] < lo) | (eda[c] > hi)).mean() * 100,
                       'media_antes': eda[c].mean(), 'media_depois': cl.mean(),
                       'desenhos_afetados': eda.loc[(eda[c] < lo) | (eda[c] > hi), 'Plano Desenho'].nunique()})
pd.DataFrame(linhas)

In [ ]:
# Os "outliers" do alvo são erro ou física? Se o tempo teórico também é enorme, é peça grande (legítimo).
cauda = eda[eda[ALVO] > eda[ALVO].quantile(.99)]
print(f'top 1% do alvo: {len(cauda):,} linhas, {cauda["Plano Desenho"].nunique()} desenhos')
print('razão real/teórico na cauda  :', cauda['razao_real_teorico'].describe()[['min', '50%', 'max']].round(2).to_dict())
print('razão real/teórico no resto  :', eda['razao_real_teorico'].describe()[['min', '50%', 'max']].round(2).to_dict())

# razão extrema (>> mediana): peças minúsculas, onde tempo fixo (perfuração/deslocamento) domina
extremos = eda[eda['razao_real_teorico'] > eda['razao_real_teorico'].quantile(.99)]
print('\ntop 1% de razão: perímetro mediano =', round(extremos['Perimetro (mm)'].median()), 'mm  vs geral =', round(eda['Perimetro (mm)'].median()), 'mm')

## 4. Correlações e interações

In [ ]:
NOMES = {'Potência Laser (W)': 'Potência do laser', 'Pressão Gás (bar)': 'Pressão do gás', 'Velocidade Corte (m/min)': 'Velocidade de corte',
         'Posição Foco (mm)': 'Posição do foco', 'Diâmetro Bocal (mm)': 'Diâmetro do bocal', 'Distância Bocal (mm)': 'Distância do bocal',
         'Espessura (mm)': 'Espessura', 'Perimetro (mm)': 'Perímetro', 'tempo_teorico_min': 'Tempo teórico (perímetro ÷ velocidade)', ALVO: 'Tempo real'}
corr_cols = PARAMS_NUM + GEOMETRIA + ['tempo_teorico_min']

pearson = eda[corr_cols + [ALVO]].corr()[ALVO].drop(ALVO)
spearman = eda[corr_cols + [ALVO]].corr(method='spearman')[ALVO].drop(ALVO)
ordem = spearman.abs().sort_values().index          # barh: maior |r| no topo

fig, axs = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, serie, titulo in zip(axs, [pearson, spearman], ['Pearson (relação linear)', 'Spearman (relação por ranking)']):
    v = serie[ordem]
    ax.barh([NOMES[c] for c in ordem], v.values, height=0.62,
            color=[COR['azul'] if x >= 0 else COR['vermelho'] for x in v.values])
    ax.axvline(0, color=COR['texto2'], lw=0.9)
    for y, x in enumerate(v.values):
        ax.text(x + (0.02 if x >= 0 else -0.02), y, f'{x:+.3f}', va='center', ha='left' if x >= 0 else 'right', fontsize=9, color=COR['texto'])
    ax.set_xlim(-1.1, 1.15); ax.set_title(titulo); ax.set_xlabel('correlação com o Tempo real  (azul = positiva, vermelho = negativa)')
    ax.grid(axis='y', visible=False)
fig.suptitle('O tempo teórico (perímetro ÷ velocidade) é, de longe, a coluna mais ligada ao tempo real', x=0.01, ha='left', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); salvar(fig, 'correlacao_com_alvo'); plt.show()

In [ ]:
# Matriz completa (Spearman): mostra também a relação entre as próprias variáveis
from matplotlib.colors import LinearSegmentedColormap
cmap_div = LinearSegmentedColormap.from_list('div', [COR['vermelho'], COR['neutro'], COR['azul']])   # vermelho (-) · cinza (0) · azul (+)

cols_m = corr_cols + [ALVO]
cm = eda[cols_m].corr(method='spearman')
fig, ax = plt.subplots(figsize=(9.5, 7.5))
im = ax.imshow(cm, cmap=cmap_div, vmin=-1, vmax=1)
rot = [NOMES[c] for c in cols_m]
ax.set_xticks(range(len(cols_m))); ax.set_xticklabels(rot, rotation=40, ha='right')
ax.set_yticks(range(len(cols_m))); ax.set_yticklabels(rot)
for i in range(len(cols_m)):
    for j in range(len(cols_m)):
        ax.text(j, i, f'{cm.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(cm.iloc[i, j]) > 0.6 else COR['texto'])
ax.grid(False); ax.spines[:].set_visible(False)
ax.set_title('Correlação de Spearman entre as colunas numéricas (treino)')
fig.colorbar(im, ax=ax, shrink=0.75, label='correlação')
plt.tight_layout(); salvar(fig, 'correlacao_matriz'); plt.show()

In [ ]:
# Tabela com os números (inclui a correlação com o alvo em log, que reduz o peso das peças enormes)
pd.DataFrame({'pearson': pearson, 'spearman': spearman,
              'pearson_log_alvo': eda[corr_cols].corrwith(np.log(eda[ALVO]))}).sort_values('spearman', key=abs, ascending=False)

**Leitura dos gráficos de correlação**

- **Tempo teórico:** Pearson 0,999 e Spearman 0,89 com o tempo real; nenhuma outra coluna chega perto. O perímetro sozinho tem 0,60 (Pearson) e 0,68 (Spearman).
- **Velocidade:** quase nada no Pearson (−0,12), mas −0,52 no Spearman. A relação é inversa e **não linear** (tempo ∝ 1/velocidade), e a correlação linear não enxerga isso.
- **Perímetro x parâmetros = 0,00 na matriz:** cada desenho foi cortado com todas as 406 combinações de parâmetros, então o tamanho da peça é independente da configuração.
- **Parâmetros correlacionados entre si** (Pressão x Foco −0,85, Foco x Distância do bocal +0,79, Espessura x Diâmetro do bocal +0,79): vêm da tabela de parâmetros e geram colinearidade, o que importa para a regressão múltipla.
- Só as colunas numéricas aparecem aqui; as categóricas (Material, Gás, Bocal, Acabamento) não têm correlação de Pearson/Spearman.

In [ ]:
# Interação central: tempo = perímetro / velocidade  (relação multiplicativa, linear só em log-log)
amostra = eda.sample(30000, random_state=0)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].scatter(amostra['Perimetro (mm)'], amostra[ALVO], s=2, alpha=.3);         ax[0].set_title('Perímetro x Tempo real (linear)')
ax[1].scatter(amostra['Velocidade Corte (m/min)'], amostra[ALVO], s=2, alpha=.3); ax[1].set_title('Velocidade x Tempo real (linear)')
ax[2].scatter(amostra['tempo_teorico_min'], amostra[ALVO], s=2, alpha=.3)
ax[2].set_xscale('log'); ax[2].set_yscale('log'); ax[2].set_title('Tempo teórico x Tempo real (log-log)')
lim = [amostra['tempo_teorico_min'].min(), amostra['tempo_teorico_min'].max()]; ax[2].plot(lim, lim, 'r--', lw=1, label='y = x'); ax[2].legend()
plt.tight_layout(); plt.show()

r_lin = eda[['tempo_teorico_min', ALVO]].corr().iloc[0, 1]
r_log = np.corrcoef(np.log(eda['tempo_teorico_min']), np.log(eda[ALVO]))[0, 1]
print(f'corr(teórico, real) linear = {r_lin:.4f} | em log-log = {r_log:.4f}')

In [ ]:
# Onde o real se afasta do teórico? razão = real / teórico, por grupo de parâmetro
eda['espessura_faixa'] = pd.cut(eda['Espessura (mm)'], [0, 2, 4, 8, 15, 30], labels=['<=2', '2-4', '4-8', '8-15', '>15'])
eda['potencia_faixa'] = eda['Potência Laser (W)'].astype(int).astype(str) + ' W'
for c in ['Material', 'Gás Auxiliar', 'Tipo Bocal', 'espessura_faixa', 'potencia_faixa', 'Tipo de Acabamento']:
    display(eda.groupby(c, observed=True)['razao_real_teorico'].agg(['count', 'median', lambda s: s.quantile(.9)]).rename(columns={'<lambda_0>': 'p90'}).round(2))

## 5. Leakage

Uma coluna é leakage se (a) **só existe depois** do corte, ou (b) é **função direta do alvo**. Na API o usuário só informa material, espessura, potência, gás e o DXF, então qualquer coluna que não dê para calcular disso **não pode ser feature**.

In [ ]:
checagens = {
    'Tempo Real Segundos == Tempo Real Minutos * 60': np.allclose(eda['Tempo Real Segundos'], eda[ALVO] * 60),
    'Diferença == Real - Estimado (usa o alvo)': np.allclose(eda['Diferença de Tempo Minutos'], eda[ALVO] - eda['Tempo Corte Estimado Minutos']),
    'Estimado == Perímetro / Velocidade (é só a fórmula)': np.allclose(eda['Tempo Corte Estimado Minutos'], eda['tempo_teorico_min']),
}
for k, v in checagens.items():
    print(f'{"SIM" if v else "NÃO"}  {k}')

# varredura automática: qualquer numérica muito correlacionada com o alvo
num_all = eda.select_dtypes('number').drop(columns=[ALVO])
varredura = pd.DataFrame({'spearman': num_all.corrwith(eda[ALVO], method='spearman'),
                          'pearson': num_all.corrwith(eda[ALVO])})
varredura['alerta'] = np.where(varredura['spearman'].abs() > 0.95, 'LEAKAGE?', '')
varredura.sort_values('spearman', key=abs, ascending=False).head(10)

In [ ]:
veredito = pd.DataFrame([
    ('Tempo Real Segundos',          'LEAKAGE',    'é o alvo em outra unidade'),
    ('Diferença de Tempo Minutos',   'LEAKAGE',    'contém o alvo (real - estimado)'),
    ('Tempo Corte Estimado Segundos','REDUNDANTE', 'é a fórmula perímetro/velocidade; já cabe em tempo_teorico_min'),
    ('Tempo Corte Estimado Minutos', 'REDUNDANTE', 'idem; não usar as duas versões'),
    ('tempo_teorico_min',            'OK',         'calculável na API (perímetro do DXF + velocidade da tabela)'),
], columns=['coluna', 'veredito', 'motivo'])
veredito

In [ ]:
# Leakage por grupo: o mesmo desenho não pode estar em treino e teste (checagem de IDs, sem olhar o alvo do teste)
ids_tr, ids_te = set(df_treino['Plano Desenho'].astype(str)), set(df_teste['Plano Desenho'].astype(str))
print('desenhos em comum treino x teste:', len(ids_tr & ids_te))

## 6. Drift: tempo e grupos

- **Tempo:** o dataset **não tem coluna de data/ordem** (é uma matriz fixa de combinações x desenhos), então não há drift temporal a medir nem split temporal a fazer.
- **Grupos:** o risco real é um desenho novo ser diferente dos vistos. Simulamos isso **dentro do treino** com `GroupKFold` por desenho: cada fold é "um conjunto de desenhos novos", comparado com o resto via KS.

In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
grupos = eda['Plano Desenho'].astype(str)
cols_drift = ['Perimetro (mm)', ALVO, 'razao_real_teorico']

linhas = []
for k, (i_resto, i_fold) in enumerate(gkf.split(eda, groups=grupos), 1):
    for c in cols_drift:
        ks = stats.ks_2samp(eda.iloc[i_fold][c], eda.iloc[i_resto][c])
        linhas.append({'fold': k, 'coluna': c, 'KS': ks.statistic, 'p': ks.pvalue,
                       'mediana_fold': eda.iloc[i_fold][c].median(), 'mediana_resto': eda.iloc[i_resto][c].median()})
drift = pd.DataFrame(linhas)
display(drift.pivot(index='coluna', columns='fold', values='KS').assign(max_KS=lambda d: d.max(axis=1)))
display(drift.groupby('coluna')[['mediana_fold', 'mediana_resto']].agg(['min', 'max']))

In [ ]:
# Parâmetros de corte: como todos os desenhos passam pelas mesmas 406 combinações, não deve haver drift
print('combinações distintas por desenho (mín/máx):',
      eda.groupby('Plano Desenho', observed=True)[PARAMS_NUM].apply(lambda g: len(g.drop_duplicates())).agg(['min', 'max']).tolist())

# Um desenho é "estável"? variação do alvo entre desenhos x dentro de um desenho
por_desenho = eda.groupby('Plano Desenho', observed=True)['razao_real_teorico'].median()
print('razão real/teórico mediana por desenho: p5 =', por_desenho.quantile(.05).round(2), '| p50 =', por_desenho.median().round(2), '| p95 =', por_desenho.quantile(.95).round(2))

# Modelagem (benchmark)

Quatro modelos, do mais simples ao mais completo:
1. Fórmula fixa (referência, sem treino)
2. Regressão linear simples
3. Regressão linear múltipla
4. Regressão polinomial múltipla

Comparação por **validação cruzada por desenho (`GroupKFold`) só no treino**. O teste é aberto uma única vez, no final.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

ALVO = 'Tempo Real Minutos'

def com_tempo_teorico(d):
    # tempo teórico (min) = perímetro (mm) / (velocidade (m/min) * 1000)
    d = d.copy()
    d['tempo_teorico_min'] = d['Perimetro (mm)'] / (d['Velocidade Corte (m/min)'] * 1000)
    return d

treino = com_tempo_teorico(df_treino)
teste = com_tempo_teorico(df_teste)

FEATURES_NUM = ['Potência Laser (W)', 'Pressão Gás (bar)', 'Velocidade Corte (m/min)', 'Posição Foco (mm)',
                'Diâmetro Bocal (mm)', 'Distância Bocal (mm)', 'Espessura (mm)', 'Perimetro (mm)', 'tempo_teorico_min']
FEATURES_CAT = ['Material', 'Gás Auxiliar']   # Tipo Bocal e Acabamento são função de Material + Gás (redundantes)

X_treino, y_treino = treino[FEATURES_NUM + FEATURES_CAT], treino[ALVO]
X_teste, y_teste = teste[FEATURES_NUM + FEATURES_CAT], teste[ALVO]
grupos_treino = treino['Plano Desenho'].astype(str)

cv = GroupKFold(n_splits=5)
resultados = {}
oof = {}   # previsões fora-da-amostra (cada linha prevista por um modelo que não viu o desenho dela)

def metricas(y, pred):
    return {'MAE': mean_absolute_error(y, pred), 'RMSE': root_mean_squared_error(y, pred),
            'MAPE_%': mean_absolute_percentage_error(y, pred) * 100, 'R2': r2_score(y, pred)}

def avaliar_cv(nome, construir_modelo=None):
    # média das métricas nos 5 folds; None = fórmula fixa (não treina, só calcula)
    folds, pred_oof = [], np.full(len(y_treino), np.nan)
    for i_tr, i_va in cv.split(X_treino, y_treino, groups=grupos_treino):
        if construir_modelo is None:
            pred = X_treino.iloc[i_va]['tempo_teorico_min']
        else:
            modelo = construir_modelo().fit(X_treino.iloc[i_tr], y_treino.iloc[i_tr])
            pred = modelo.predict(X_treino.iloc[i_va])
        pred_oof[i_va] = np.asarray(pred)
        folds.append(metricas(y_treino.iloc[i_va], pred))
    oof[nome] = pd.Series(pred_oof, index=y_treino.index)
    resultados[nome] = pd.DataFrame(folds).mean().to_dict()
    return pd.DataFrame([resultados[nome]], index=[nome])

## 04.01 Fórmula fixa

`tempo = perímetro / velocidade`. Não tem treino: é a referência que os modelos precisam superar.

In [ ]:
avaliar_cv('1. Fórmula fixa')

## 04.02 Regressão linear simples

Uma única variável: `tempo_teorico_min`.

In [ ]:
def linear_simples():
    return Pipeline([
        ('colunas', ColumnTransformer([('x', 'passthrough', ['tempo_teorico_min'])])),
        ('modelo', LinearRegression()),
    ])

avaliar_cv('2. Linear simples', linear_simples)

## 04.03 Regressão linear múltipla

Parâmetros de corte numéricos (padronizados) + Material e Gás (one-hot).

In [ ]:
def pre_processador(grau=None):
    if grau is None:
        numericas = StandardScaler()
    else:
        numericas = Pipeline([('escala', StandardScaler()), ('polinomio', PolynomialFeatures(degree=grau, include_bias=False))])
    return ColumnTransformer([
        ('num', numericas, FEATURES_NUM),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), FEATURES_CAT),
    ])

def linear_multipla():
    return Pipeline([('pre', pre_processador()), ('modelo', LinearRegression())])

avaliar_cv('3. Linear múltipla', linear_multipla)

## 04.04 Regressão polinomial múltipla (grau 2)

As mesmas variáveis, com quadrados e interações entre as numéricas.

In [ ]:
def polinomial_multipla():
    return Pipeline([('pre', pre_processador(grau=2)), ('modelo', LinearRegression())])

avaliar_cv('4. Polinomial múltipla (grau 2)', polinomial_multipla)

## 04.05 Comparação (validação cruzada no treino)

In [ ]:
pd.DataFrame(resultados).T

## Gráficos de erro

Usam as **previsões fora-da-amostra do treino** (`oof`): cada linha é prevista por um modelo que **não viu o desenho dela**. O teste continua fechado até a avaliação final.

In [ ]:
CURTOS = {'1. Fórmula fixa': 'Fórmula fixa', '2. Linear simples': 'Linear simples',
          '3. Linear múltipla': 'Linear múltipla', '4. Polinomial múltipla (grau 2)': 'Polinomial múltipla'}
PISO = 1e-3     # previsões lineares podem ficar <= 0 em peças minúsculas; só para exibir em escala log
print('previsões <= 0 (exibidas no piso):', {CURTOS[n]: int((p <= 0).sum()) for n, p in oof.items()})

tab = pd.DataFrame(resultados).T
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, (met, titulo, fmt) in zip(axs, [('MAE', 'Erro médio absoluto (minutos)', '{:.2f}'), ('RMSE', 'RMSE (minutos)', '{:.2f}'), ('MAPE_%', 'Erro percentual médio (%)', '{:.0f}%')]):
    nomes = [CURTOS[n] for n in tab.index]
    barras = ax.bar(nomes, tab[met].values, width=0.62, color=[COR['cinza']] + [COR['azul']] * (len(nomes) - 1))
    for b, v in zip(barras, tab[met].values):
        ax.text(b.get_x() + b.get_width() / 2, v, fmt.format(v), ha='center', va='bottom', fontsize=10, color=COR['texto'])
    ax.set_title(titulo); ax.tick_params(axis='x', rotation=22); ax.grid(axis='x', visible=False); ax.set_ylim(0, tab[met].max() * 1.15)
fig.suptitle('Calibrar o tempo teórico corta o erro em ~80%; modelos mais complexos quase não melhoram', x=0.01, ha='left', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); salvar(fig, 'erro_comparacao_modelos'); plt.show()

In [ ]:
# Real x previsto: quanto mais perto da diagonal, melhor
amostra = y_treino.sample(25000, random_state=0).index
lim = [0.002, 400]
fig, axs = plt.subplots(1, 4, figsize=(17, 4.4), sharex=True, sharey=True)
for ax, (nome, pred) in zip(axs, oof.items()):
    ax.scatter(y_treino.loc[amostra], pred.loc[amostra].clip(lower=PISO), s=2, alpha=0.15, color=COR['azul'] if nome != '1. Fórmula fixa' else COR['cinza'], rasterized=True)
    ax.plot(lim, lim, color=COR['texto'], lw=1, ls='--')
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(CURTOS[nome]); ax.set_xlabel('Tempo real (min)')
axs[0].set_ylabel('Tempo previsto (min)')
fig.suptitle('Real x previsto: os modelos colam na diagonal, mas não preveem menos que ~0,17 min (peças muito pequenas ficam acima)', x=0.01, ha='left', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); salvar(fig, 'erro_real_vs_previsto'); plt.show()

In [ ]:
# Quanto cada modelo erra "em vezes": previsto ÷ real (0 = acerto exato; ±0,3 = a metade / o dobro)
bins = np.linspace(-1.2, 1.2, 80)
fig, axs = plt.subplots(1, 4, figsize=(17, 4), sharex=True, sharey=True)
for ax, (nome, pred) in zip(axs, oof.items()):
    r = np.log10(pred.clip(lower=PISO) / y_treino)
    ax.hist(r.clip(-1.2, 1.2), bins=bins, color=COR['cinza'] if nome == '1. Fórmula fixa' else COR['azul'])
    ax.axvline(0, color=COR['texto'], lw=1, ls='--')
    ax.text(0.97, 0.93, f'mediana: {10 ** r.median():.2f}× o real', transform=ax.transAxes, ha='right', va='top', fontsize=10, color=COR['texto'])
    ax.set_title(CURTOS[nome])
    ax.set_xticks([-1, -np.log10(2), 0, np.log10(2), 1]); ax.set_xticklabels(['10× menor', '2× menor', 'acerto', '2× maior', '10× maior'], rotation=30)
    ax.set_yticks([])
fig.suptitle('A fórmula fixa subestima o tempo na quase totalidade das linhas; os modelos treinados ficam perto do acerto', x=0.01, ha='left', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); salvar(fig, 'erro_distribuicao_razao'); plt.show()

In [ ]:
# Onde o erro está? Por tamanho da peça (faixas do perímetro, que é uma entrada conhecida; 5 faixas com o mesmo nº de linhas)
faixa = pd.qcut(treino['Perimetro (mm)'], 5)
rotulos = [f'{iv.left / 1000:.1f}–{iv.right / 1000:.1f} m' for iv in faixa.cat.categories]
por_faixa = {'MAE': {}, 'MAPE': {}}
for nome, pred in oof.items():
    df_e = pd.DataFrame({'real': y_treino, 'pred': pred, 'faixa': faixa})
    df_e['abs'] = (df_e['pred'] - df_e['real']).abs()
    g = df_e.groupby('faixa', observed=True)
    por_faixa['MAE'][nome] = g['abs'].mean().values
    por_faixa['MAPE'][nome] = (g.apply(lambda d: (d['abs'] / d['real']).mean()) * 100).values

estilos = {'1. Fórmula fixa': (COR['cinza'], '--'), '2. Linear simples': (COR['azul'], '-'),
           '3. Linear múltipla': (COR['laranja'], '-'), '4. Polinomial múltipla (grau 2)': (COR['verde'], '-')}
fig, axs = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, (met, titulo) in zip(axs, [('MAE', 'Erro médio absoluto (minutos, escala log)'), ('MAPE', 'Erro percentual médio (%)')]):
    for nome, valores in por_faixa[met].items():
        cor, ls = estilos[nome]
        ax.plot(rotulos, valores, color=cor, ls=ls, lw=2, marker='o', ms=7, mfc=cor, mec='white', mew=1.5, label=CURTOS[nome])
    ax.set_title(titulo); ax.set_xlabel('Perímetro da peça (faixas de mesmo tamanho: menores → maiores)')
    ax.tick_params(axis='x', rotation=15)
axs[0].set_yscale('log')
from matplotlib.ticker import NullFormatter
axs[0].set_yticks([0.1, 0.2, 0.5, 1, 2, 3]); axs[0].set_yticklabels(['0,1', '0,2', '0,5', '1', '2', '3']); axs[0].yaxis.set_minor_formatter(NullFormatter())
axs[1].legend(frameon=False, loc='upper right')
fig.suptitle('Em valor absoluto o erro cresce com a peça; em percentual, o problema são as peças pequenas', x=0.01, ha='left', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); salvar(fig, 'erro_por_tamanho'); plt.show()
pd.DataFrame(por_faixa['MAPE'], index=rotulos).rename(columns=CURTOS).round(1).rename_axis('MAPE (%) por faixa de perímetro')

**Leitura dos gráficos de erro** (previsões fora-da-amostra no treino)

- **Fórmula fixa:** subestima o tempo em ~91% das linhas (mais de 1,5× abaixo do real); mediana de 0,37× o real. Erro médio de 0,81 min e RMSE de 2,75 min.
- **Modelos treinados:** erro médio de ~0,17 min (queda de ~79%), RMSE de ~0,39 min, centrados em ~1,1× o real. Linear simples, múltipla e polinomial são praticamente iguais.
- **Onde ainda erram:** nas peças pequenas. As três previsões lineares têm um piso de ~0,17 min (mínimo previsto) e o real chega a 0,002 min, então peças muito rápidas são superestimadas. Cerca de 11% a 15% das linhas são previstas acima de 1,5× o real e ~7% abaixo de 1/1,5.
- **Por tamanho (MAPE):** na faixa das menores peças, linear simples 46%, múltipla 52% e polinomial 56%; na das maiores, 13%, 12% e 11%. Os modelos mais complexos ganham um pouco nas peças grandes e perdem nas pequenas.
- **Próximos refinos:** alvo em `log`, interações do tempo teórico com Gás/Material/espessura e um termo para o tempo fixo.

## 04.06 Avaliação final no teste

**Aberta uma única vez**, depois de definidos os modelos. Cada um é treinado no treino inteiro.

In [ ]:
modelos_finais = {'1. Fórmula fixa': None, '2. Linear simples': linear_simples,
                  '3. Linear múltipla': linear_multipla, '4. Polinomial múltipla (grau 2)': polinomial_multipla}
final = {}
for nome, construir in modelos_finais.items():
    if construir is None:
        pred = X_teste['tempo_teorico_min']
    else:
        pred = construir().fit(X_treino, y_treino).predict(X_teste)
    final[nome] = metricas(y_teste, pred)
pd.DataFrame(final).T